In [1]:
# ==========================================
# Load Training Dataset
# ==========================================

with open(
    "data/sherlock.txt",
    "r",
    encoding="cp1252"
) as f:
    raw_data = f.read()

print("Total characters:", len(raw_data))

Total characters: 562202


In [2]:
# ==========================================
# Build BPE Tokenizer
# ==========================================

from tokenizer import build_tokenizer

tokenizer, vocab, merge_rank = build_tokenizer(
    raw_data,
    num_merges=150
)

print("Vocabulary size:", len(vocab))

Vocabulary size: 239


In [3]:
# ==========================================
# Create Dataset and DataLoader
# ==========================================

from data import DataLoader

loader = DataLoader(
    raw_data,
    tokenizer,
    batch_size=8,
    length=16,
    stride=4
)

In [7]:
loader.reset()

embedding_inputs = []
target_outputs = []

while True:

    x, y = loader.get_batch()

    if x is None:
        break

    embedding_inputs.append(x)
    target_outputs.append(y)

print("Number of batches:", len(embedding_inputs))

if len(embedding_inputs) > 0:

    print(
        "Input batch shape:",
        np.asarray(embedding_inputs[0]).shape
    )

    print(
        "Target batch shape:",
        np.asarray(target_outputs[0]).shape
    )

Number of batches: 11269
Input batch shape: (8, 16)
Target batch shape: (8, 16)


In [8]:
# ==========================================
# Create GPT Model
# ==========================================

import numpy as np

from model import GPT


embedding_matrix = (
    np.random.randn(
        len(vocab),
        256
    ) * 0.02
)

gpt = GPT(
    num_layers=1,
    embed_dim=256,
    num_heads=4,
    sequence_length=16,
    embedding_matrix=embedding_matrix
)

print("GPT model created")

GPT model created


In [9]:
# ==========================================
# Load Trained Weights
# ==========================================

gpt.load_weights(
    "gpt_stride4_epoch_15.pkl"
)

print(
    "Embedding shape:",
    gpt.embedding_matrix.shape
)

print(
    "Position embedding shape:",
    gpt.position_embeddings.shape
)

print(
    "W_out shape:",
    gpt.W_out.shape
)

Weights loaded from: gpt_stride4_epoch_15.pkl
Embedding shape: (239, 256)
Position embedding shape: (16, 256)
W_out shape: (256, 239)


In [10]:
# ==========================================
# Evaluate Trained Model
# ==========================================

from evaluate import evaluate_model

metrics = evaluate_model(
    gpt,
    tokenizer,
    embedding_inputs,
    target_outputs,
    max_batches=100
)

print(
    "Overall Accuracy:",
    metrics["overall_accuracy"]
)

print(
    "Non-Space Accuracy:",
    metrics["non_space_accuracy"]
)

print(
    "Space Prediction Percentage:",
    metrics["space_prediction_percentage"]
)

Overall Accuracy: 0.412265625
Non-Space Accuracy: 0.24351446567737803
Space Prediction Percentage: 0.347265625


In [11]:
# ==========================================
# Generate Text
# ==========================================

from generate import generate

prompts = [
    "Sherlock Holmes was ",
    "Holmes said ",
    "The man ",
    "I was ",
    "It was "
]

for prompt in prompts:

    generated = generate(
        gpt,
        tokenizer,
        prompt,
        max_new_tokens=30,
        temperature=0.8,
        do_sample=True,
        top_k=10
    )

    print("Prompt:", repr(prompt))
    print("Generated:", repr(generated))
    print()

Prompt: 'Sherlock Holmes was '
Generated: 'Sherlock Holmes was a servant before we say to\ntell me, and I '

Prompt: 'Holmes said '
Generated: 'Holmes said he,\nperfeet\nmight of person easy evidents the '

Prompt: 'The man '
Generated: 'The man of a listened in her father. If she might have been'

Prompt: 'I was '
Generated: 'I was not thill the day that I would be find that the holl'

Prompt: 'It was '
Generated: 'It was a\npreput the part of them of the servate tho'



In [12]:
text = generate(
    gpt,
    tokenizer,
    "Sherlock Holmes was ",
    max_new_tokens=30,
    do_sample=False,
    top_k=10
)

print(repr(text))

'Sherlock Holmes was a small came to be a part of the disappeared'
